Trains a chess resnet

In [ ]:
# Colab Setup: clone repo and install requirements
import os, sys

# check if running in Colab
IN_COLAB = "google.colab" in sys.modules
print(f"Running on Colab:{IN_COLAB}")

if IN_COLAB:
    if not os.path.exists("vis10n"):
        print("Cloning full repo")
        !git clone https://github.com/simchaorgel/vis10n.git
    !cd vis10n && git pull
    sys.path.append("vis10n")
    # Download requirements
    print("Downloading requirements")
    !pip install -q -r vis10n/requirements.txt
else:
    # Running locally from training/: make the repo root importable
    sys.path.append("..")

In [ ]:
# Main imports
import torch as t
from dataclasses import dataclass
from torch.utils.data import Subset, DataLoader
import torch.nn.functional as F
from tqdm.notebook import tqdm
import wandb

In [ ]:
# Local imports: this cell imports the necessary files
# either locally or from colabs repo clone
from models.chess_resnet import ChessResnet
import optimizers
from dataloaders.lichess import get_lichess

In [ ]:
# Check if we have access to a GPU
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Wandb login
wandb.login()

In [ ]:
# Load data
def colab_get_lichess(trainset_size: int, testset_size: int):
    """On Colab, copies the processed data from Drive into the repo's data folder first"""
    if IN_COLAB:
        processed_dir = "/content/vis10n/data/chess/processed"
        # Only copy once per session (cell may be re-run)
        if not os.path.exists(f"{processed_dir}/boards.npy"):
            from google.colab import drive
            drive.mount('/content/drive')
            !mkdir -p {processed_dir}
            !cp /content/drive/MyDrive/data/chess/processed/*.npy {processed_dir}/

    return get_lichess(trainset_size, testset_size)

In [ ]:
# Training args
#####################################################
@dataclass
class ChessResnetTrainingArgs:
    trainset_size: int = 900_000
    testset_size: int = 100_000
    batch_size: int = 512
    epochs: int = 5
    optimizer: str = "AdamW"
    learning_rate: float = 1e-3
    weight_decay: float = 0
    save_model: bool = True

In [ ]:
# Train
#####################################################
def train(args: ChessResnetTrainingArgs) -> ChessResnet:
    """
    Trains ResNet on MNIST, logging to wandb. Returns the trained model, also saves to wandb
    """
    try:
        # Wandb init
        wandb.init(entity="simchao", project="chess-resnet", config=args)
        # Create ResNet instance
        model = ChessResnet().to(device)
        #wandb.watch(model, log="all", log_freq=50)
        # Get data
        trainset, testset = colab_get_lichess(args.trainset_size, args.testset_size)
        train_data = DataLoader(trainset, batch_size=args.batch_size, shuffle=True)
        test_data = DataLoader(testset, batch_size=args.batch_size, shuffle=False)
        # Define optimizer
        optimizer = optimizers.get_optimizer(args.optimizer)(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
        # Logging lists
        examples_seen = 0

        # For every epoch:
        for epoch in range(args.epochs):
            epoch_accurate_predictions = 0 # For validating
            pbar = tqdm(train_data)

            # Train - loop for every batch in this epoch
            model.train()
            for boards, moves in pbar:
                # Move data to device, perform forward pass
                boards, moves = boards.to(device).float(), moves.to(device)
                logits = model(boards)
                # Compute loss, perform backward pass
                loss = F.cross_entropy(logits, moves)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                # Update logs & progress bar
                examples_seen += boards.shape[0]
                wandb.log({"loss": loss.item()}, step=examples_seen)
                pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

            # Test to get accuracy
            model.eval()
            for boards, moves in test_data:
                boards, moves = boards.to(device).float(), moves.to(device)
                with t.inference_mode():
                    logits = model(boards)

                # Calculate accuracy
                # Get max logit, which is treated as the models guess
                predictions = t.argmax(logits, dim=1)
                # Create tensor with 1 if the prediction was correct, 0 if not, for every img
                accuracy_bools = predictions == moves
                # Number of accurate guesses
                epoch_accurate_predictions += accuracy_bools.sum().item()

            # Before completing the epoch
            # Add accuracy percentage to list
            epoch_accuracy = epoch_accurate_predictions / len(testset)
            wandb.log({"accuracy": epoch_accuracy}, step=examples_seen)

        if args.save_model:
            t.save(model.state_dict(), "chess_resnet.pt")
            artifact = wandb.Artifact("chess_resnet", type="model")
            artifact.add_file("chess_resnet.pt")
            wandb.log_artifact(artifact)
            
        wandb.finish()
        return model
    
    except Exception:
        wandb.finish()
        raise

In [ ]:
# Run a standard train
###################################################
args = ChessResnetTrainingArgs()
model = train(args)